<a href="https://colab.research.google.com/github/richasachdev202/Project1/blob/main/Project4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.feature_selection import SelectKBest, mutual_info_regression

# -----------------------------
# 1. Paths and configuration
# -----------------------------
BASE_DIR = Path('/content/Processed_Flipdata - Processed_Flipdata.csv').resolve().parent
DATA_FILE = BASE_DIR / "Processed_Flipdata - Processed_Flipdata.csv"
OUTPUT_DIR = BASE_DIR / "mobile_price_outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

RANDOM_STATE = 42
TEST_SIZE = 0.20

# -----------------------------
# 2. Load and explore data
# -----------------------------
df = pd.read_csv(DATA_FILE)

# Remove automatically generated index column if present
unnamed_cols = [c for c in df.columns if c.lower().startswith("unnamed")]
df = df.drop(columns=unnamed_cols, errors="ignore")

# Normalize column names expected by this project
required_cols = [
    "Model", "Colour", "Memory", "RAM", "Battery_", "Rear Camera",
    "Front Camera", "AI Lens", "Mobile Height", "Processor_", "Prize"
]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing expected columns: {missing}")

# Price is the target; remove commas and convert to numeric
df["Prize"] = (
    df["Prize"].astype(str).str.replace(",", "", regex=False).str.strip()
)
df["Prize"] = pd.to_numeric(df["Prize"], errors="coerce")

# Convert camera strings such as 13MP to numeric MP
for col in ["Rear Camera", "Front Camera"]:
    df[col] = (
        df[col].astype(str)
        .str.extract(r"([0-9]+(?:\.[0-9]+)?)", expand=False)
    )
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Numeric columns
for col in ["Memory", "RAM", "Battery_", "AI Lens", "Mobile Height"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Remove rows where target is unavailable
before = len(df)
df = df.dropna(subset=["Prize"]).copy()
removed = before - len(df)

print("Dataset shape:", df.shape)
print("Rows removed because target was missing:", removed)
print("\nMissing values:\n", df.isna().sum())
print("\nData types:\n", df.dtypes)
print("\nPrice summary:\n", df["Prize"].describe())

# Save basic data-quality summary
quality = pd.DataFrame({
    "column": df.columns,
    "dtype": [str(df[c].dtype) for c in df.columns],
    "missing_count": [df[c].isna().sum() for c in df.columns],
    "unique_count": [df[c].nunique(dropna=True) for c in df.columns],
})
quality.to_csv(OUTPUT_DIR / "data_quality_summary.csv", index=False)

# -----------------------------
# 3. Outlier inspection
# -----------------------------
Q1 = df["Prize"].quantile(0.25)
Q3 = df["Prize"].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR
outlier_count = ((df["Prize"] < lower) | (df["Prize"] > upper)).sum()
print(f"\nPotential price outliers by IQR rule: {outlier_count}")

plt.figure(figsize=(8, 5))
plt.boxplot(df["Prize"].dropna())
plt.title("Mobile Phone Price – Outlier Inspection")
plt.ylabel("Price")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "price_outliers.png", dpi=180)
plt.close()

# -----------------------------
# 4. Feature extraction / selection
# -----------------------------
X = df.drop(columns=["Prize"])
y = df["Prize"]

numeric_features = [
    "Memory", "RAM", "Battery_", "Rear Camera", "Front Camera",
    "AI Lens", "Mobile Height"
]
categorical_features = ["Model", "Colour", "Processor_"]

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipe, numeric_features),
    ("cat", categorical_pipe, categorical_features),
], remainder="drop")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
)

# Transform data for feature-selection analysis
X_train_t = preprocessor.fit_transform(X_train)
X_test_t = preprocessor.transform(X_test)

feature_names = preprocessor.get_feature_names_out()

# Mutual-information feature extraction
mi = mutual_info_regression(X_train_t, y_train, random_state=RANDOM_STATE)
mi_df = pd.DataFrame({"feature": feature_names, "mutual_information": mi})
mi_df = mi_df.sort_values("mutual_information", ascending=False)
mi_df.to_csv(OUTPUT_DIR / "feature_extraction_mutual_information.csv", index=False)

print("\nTop 20 extracted features by mutual information:")
print(mi_df.head(20).to_string(index=False))

plt.figure(figsize=(9, 6))
top_mi = mi_df.head(15).sort_values("mutual_information")
plt.barh(top_mi["feature"], top_mi["mutual_information"])
plt.title("Top Features – Mutual Information")
plt.xlabel("Mutual Information")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "top_features_mutual_information.png", dpi=180)
plt.close()

# -----------------------------
# 5. Model building
# -----------------------------
model = RandomForestRegressor(
    n_estimators=400,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    min_samples_leaf=2,
)

model.fit(X_train_t, y_train)
y_pred = model.predict(X_test_t)

# -----------------------------
# 6. Model evaluation
# -----------------------------
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

metrics = pd.DataFrame({
    "Metric": ["MAE", "RMSE", "R2"],
    "Value": [mae, rmse, r2],
})
metrics.to_csv(OUTPUT_DIR / "model_metrics.csv", index=False)
print("\nModel evaluation:")
print(metrics.to_string(index=False))

# Actual vs predicted
predictions = pd.DataFrame({
    "Actual_Price": y_test.values,
    "Predicted_Price": y_pred,
})
predictions.to_csv(OUTPUT_DIR / "actual_vs_predicted.csv", index=False)

plt.figure(figsize=(7, 6))
plt.scatter(y_test, y_pred, alpha=0.65)
min_v = min(y_test.min(), y_pred.min())
max_v = max(y_test.max(), y_pred.max())
plt.plot([min_v, max_v], [min_v, max_v])
plt.xlabel("Actual Price")
plt.ylabel("Predicted Price")
plt.title("Actual vs Predicted Mobile Phone Price")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "actual_vs_predicted.png", dpi=180)
plt.close()

# -----------------------------
# 7. Feature importance analysis
# -----------------------------
importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": model.feature_importances_,
}).sort_values("importance", ascending=False)
importance_df.to_csv(OUTPUT_DIR / "feature_importance.csv", index=False)

print("\nTop 20 model features:")
print(importance_df.head(20).to_string(index=False))

plt.figure(figsize=(9, 6))
top_imp = importance_df.head(15).sort_values("importance")
plt.barh(top_imp["feature"], top_imp["importance"])
plt.title("Top Features – Random Forest Importance")
plt.xlabel("Importance")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "top_feature_importance.png", dpi=180)
plt.close()

# -----------------------------
# 8. Business-level feature summary
# -----------------------------
# Aggregate one-hot model importance back to the original feature family.
def original_feature(transformed_name):
    if transformed_name.startswith("num__"):
        return transformed_name.replace("num__", "")
    if transformed_name.startswith("cat__"):
        remainder = transformed_name.replace("cat__", "")
        for base in categorical_features:
            if remainder.startswith(base + "_") or remainder == base:
                return base
    return transformed_name

importance_df["feature_family"] = importance_df["feature"].map(original_feature)
family_importance = (
    importance_df.groupby("feature_family", as_index=False)["importance"]
    .sum().sort_values("importance", ascending=False)
)
family_importance.to_csv(OUTPUT_DIR / "feature_family_importance.csv", index=False)

# -----------------------------
# 9. Final recommendation file
# -----------------------------
recommendations = [
    "Use the model-derived feature importance to prioritize product attributes for pricing analysis.",
    "Validate the highest-ranked features with business stakeholders before changing pricing strategy.",
    "Monitor model MAE/RMSE and retrain when new mobile-phone pricing data becomes available.",
    "Use feature-family importance for management communication and detailed one-hot importance for technical analysis.",
]
pd.DataFrame({"Recommendations": recommendations}).to_csv(
    OUTPUT_DIR / "recommendations.csv", index=False
)

print("\nOutputs saved in:", OUTPUT_DIR)
print("Project completed successfully.")

Dataset shape: (541, 11)
Rows removed because target was missing: 0

Missing values:
 Model            0
Colour           0
Memory           0
RAM              0
Battery_         0
Rear Camera      0
Front Camera     0
AI Lens          0
Mobile Height    0
Processor_       0
Prize            0
dtype: int64

Data types:
 Model             object
Colour            object
Memory             int64
RAM                int64
Battery_           int64
Rear Camera        int64
Front Camera       int64
AI Lens            int64
Mobile Height    float64
Processor_        object
Prize              int64
dtype: object

Price summary:
 count      541.000000
mean     16228.375231
std      10793.588322
min        920.000000
25%       9290.000000
50%      13499.000000
75%      19280.000000
max      80999.000000
Name: Prize, dtype: float64

Potential price outliers by IQR rule: 25

Top 20 extracted features by mutual information:
                                           feature  mutual_information
     

# New Section